# Geometry and controls

This notebook separates identity-exit radii, detector failures, and censored rays; probes the local embedding metric; and compares the anchor with unrelated generated seeds. Use the same environment as notebook 01. These are exploratory synthetic-face experiments, not a biometric accuracy evaluation.


In [ ]:
from pathlib import Path
import sys
import os
ROOT = Path.cwd()
if not (ROOT / "legacy.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "legacy.py").exists():
    raise RuntimeError("Start Jupyter from the StyleGAN3 root or notebooks directory")
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs" / ".matplotlib"))
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Video
from experiments.face_boundary import *
torch.set_num_threads(4)
print("Python:", sys.executable)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())


In [ ]:
network = download_network()
generator = load_generator(network)
matcher = load_matcher()
latent = LatentSlice(generator, center="mean", dimensions=2, basis_kind="pca")
oracle = IdentityOracle(latent, matcher)
OUTPUT = ROOT / "outputs" / "geometry-mean"
OUTPUT.mkdir(parents=True, exist_ok=True)


## Threshold sweep on one fixed slice

Reusing the same anchor and axes makes thresholds comparable. The first threshold is the upstream buffalo_l default, cosine similarity >= 0.45 (distance <= 0.55); the others are stricter controls. These are not locally calibrated false-match operating points. No-face rays and radius-limit rays are kept out of the observed identity-radius summary. Radii marked censored are lower bounds, not boundary measurements.


In [ ]:
thresholds = [DEFAULT_IDENTITY_SIMILARITY, 0.6, 0.8]
angles = np.linspace(0, 2*np.pi, 16, endpoint=False)
directions = np.column_stack([np.cos(angles), np.sin(angles)])
sweep = {}
for threshold in thresholds:
    sweep[threshold] = radial_scan(oracle, directions, threshold, increment=0.25)
    values = [item["radius"] for item in sweep[threshold] if item["reason"] == "identity"]
    print(threshold, "identity exits:", len(values), "median:", np.median(values) if values else None,
          "censored:", sum(item["censored"] for item in sweep[threshold]),
          "detector exits:", sum(item["reason"] not in ("identity", "radius_limit") for item in sweep[threshold]))
figure, axes = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(6, 5))
for threshold, results in sweep.items():
    radii = [item["radius"] if item["reason"] == "identity" else np.nan for item in results]
    axes.plot(np.append(angles, angles[0]), np.append(radii, radii[0]), label=f"cos >= {threshold}")
axes.legend(loc="upper right", bbox_to_anchor=(1.4, 1.2))
axes.set_title("Observed identity-exit radii")
figure.savefig(OUTPUT / "threshold_radii.png", dpi=160, bbox_inches="tight")
plt.show()
(OUTPUT / "threshold_sweep.json").write_text(json.dumps(
    {"directions": directions.tolist(), "results": sweep, "provenance": provenance(oracle, network)}, indent=2))


## Is the local region approximately elliptical?

For normalized embeddings, `1 - cosine ≈ 0.5 * delta.T @ metric @ delta`, where `metric = J.T @ J`. The eigenvectors describe locally sensitive directions; large eigenvalues imply short local radii. Estimate at several epsilons because alignment and quantization can disrupt derivatives. This approximation is local; it need not predict contours at cosine 0.4 or 0.6.


In [ ]:
metrics = {}
for epsilon in [0.025, 0.05, 0.1]:
    metric = local_metric(oracle, 2, epsilon)
    metrics[epsilon] = metric
    eigenvalues = np.linalg.eigvalsh(metric)
    print("epsilon", epsilon, "eigenvalues", eigenvalues,
          "local axis ratio", np.sqrt(eigenvalues[-1]/max(eigenvalues[0], 1e-12)))
np.savez_compressed(OUTPUT / "local_metrics.npz",
                    **{f"epsilon_{epsilon}": metric for epsilon, metric in metrics.items()},
                    anchor=latent.anchor, basis=latent.basis)


## Eight-dimensional first exits

A 2D slice can miss weak or strong directions. Random unit rays in eight standardized W coordinates probe a larger subspace. This still does not measure the full 512D region or its volume. Billiards work in this subspace too; a 2D projection of their path would not be the same as the true 2D acceptance contour.


In [ ]:
high_latent = LatentSlice(generator, center="mean", dimensions=8, basis_kind="pca")
high_oracle = IdentityOracle(high_latent, matcher)
random = np.random.default_rng(101)
high_directions = random.standard_normal((32, 8))
high_directions /= np.linalg.norm(high_directions, axis=1, keepdims=True)
high_results = radial_scan(high_oracle, high_directions, threshold=DEFAULT_IDENTITY_SIMILARITY, increment=0.25)
identity_radii = [item["radius"] for item in high_results if item["reason"] == "identity"]
plt.hist(identity_radii, bins=10)
plt.xlabel("First-exit radius, standardized W coordinates")
plt.ylabel("Rays")
plt.title("8D slice: observed identity exits only")
plt.savefig(OUTPUT / "8d_radii.png", dpi=160)
plt.show()
print("Censored:", sum(item["censored"] for item in high_results))
print("Reasons:", {reason: sum(item["reason"] == reason for item in high_results)
                   for reason in {item["reason"] for item in high_results}})
(OUTPUT / "8d_rays.json").write_text(json.dumps(
    {"directions": high_directions.tolist(), "results": high_results,
     "provenance": provenance(high_oracle, network)}, indent=2))
np.savez_compressed(OUTPUT / "8d_slice.npz", anchor=high_latent.anchor, basis=high_latent.basis)


## Unrelated-seed controls

Generate independent Gaussian Z samples and compare them with the anchor. Independent seeds are not guaranteed independent identities. These scores are a synthetic negative control; choosing a calibrated false-match rate needs held-out labeled identities. Detection failures are excluded and counted explicitly.


In [ ]:
from PIL import Image
control_seeds = list(range(100, 132))
control_scores = []
invalid_controls = []
with torch.inference_mode():
    for seed in control_seeds:
        sampled_z = np.random.RandomState(seed).randn(1, generator.z_dim)
        styles = latent._map(sampled_z)
        image = generator.synthesis(styles, noise_mode="const", force_fp32=True)
        rgb = (image[0].permute(1, 2, 0)*127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
        observed = face_embedding(matcher, rgb)
        if observed.valid:
            control_scores.append(float(observed.embedding @ oracle.reference_observation.embedding))
        else:
            invalid_controls.append({"seed": seed, "reason": observed.reason})
plt.hist(control_scores, bins=12)
for threshold in thresholds:
    plt.axvline(threshold, linestyle="--", label=f"tau={threshold}")
plt.xlabel("Cosine similarity to anchor")
plt.ylabel("Generated seeds")
plt.legend()
plt.savefig(OUTPUT / "unrelated_seed_scores.png", dpi=160)
plt.show()
print("Invalid controls:", invalid_controls)
(OUTPUT / "controls.json").write_text(json.dumps(
    {"seeds": control_seeds, "scores": control_scores, "invalid": invalid_controls}, indent=2))


## Optional multi-anchor replication

This is intentionally off by default because every additional anchor adds many synthesis and recognition calls. Holding the basis seed fixed gives the same PCA axes across anchors. Compare the same number of directions and retain censored outcomes. Repeat later with random standardized W directions and Z.


In [ ]:
RUN_MULTI_ANCHOR = False
if RUN_MULTI_ANCHOR:
    replications = {}
    for seed in [11, 23, 42, 77]:
        try:
            repeated = IdentityOracle(LatentSlice(generator, center="seed", seed=seed, dimensions=2), matcher)
            replications[seed] = radial_scan(repeated, directions, threshold=DEFAULT_IDENTITY_SIMILARITY, increment=0.25)
        except ValueError as error:
            replications[seed] = {"error": str(error)}
    (OUTPUT / "multi_anchor.json").write_text(json.dumps(replications, indent=2))
    print(replications)


For a next study, compare several matchers on identical rendered frames, characterize full ray profiles for exits and reentries, and match both path length and evaluation budget across samplers. Use perceptual or aligned-pixel diversity alongside embedding diversity to distinguish nuisance changes from changes the matcher considers identity-relevant.
